## Testing the new parser code to load models with different options

In [1]:
from pycbc.workflow import WorkflowConfigParser
from pycbc.inference.models.gaussian_noise import GaussianNoise
from pycbc.inference.models.marginalized_gaussian_noise import MarginalizedHMPhase
from pycbc.inference.models import read_from_config
import h5py
import numpy as np
import matplotlib.pyplot as plt

/Users/vikasjadhav/anaconda3/envs/hmnumerical/lib/python3.11/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1


In [2]:

def create_parser(
        injection_file,variable_params_file,data_file,model_file,
        mode_array,**kwargs):
    """ 
    Create a Workflowconfigparser that updates static_params by reading it from 
    the injection file. 
    Inputs : 
    "injection file" : an hdf file containing only one injection
    "variable_params_file" : a config file containing section of variable_params
    and must include the priors of each parameter.
    "data_file" : a config file containing common data settings.The trigger-time and 
    injection-file options will be updated using the provided injection.
    "model_file" : config file with the model settings.
    "mode_array" : mode array to use in the signal model.
    "kwargs" : additional options for the [model] section.
    Output : 
    "cp" : a workflow config parser
    """
    cp = WorkflowConfigParser([data_file,variable_params_file,model_file])
    
    
    ## Read the injection file to gather all the params
    inj = h5py.File(injection_file,'r')
    all_params = {}
    for p in inj.keys():
        all_params[p] = inj[p][0]
    for sp in inj.attrs['static_args']:
        all_params[sp] = inj.attrs[sp]
    static_params = all_params.copy()
    ## Remove the params included in the variable params
    for vp in cp['variable_params'].keys():
        _ = static_params.pop(vp)
    
    ## Add the mode array to be used in the model
    static_params['mode_array'] = mode_array
    ## Create and add options to static_params section
    cp.add_section('static_params')
    for sp in static_params:
        cp.add_options_to_section('static_params',
                                  [(sp,str(static_params[sp]))])
    
    ## Update options to data section
    cp.add_options_to_section('data',[('trigger-time',str(static_params['tc'])),
                                      ('injection-file',injection_file)])
    for key, value in kwargs.items():
        cp.add_options_to_section('model', [(key, str(value))])
    return cp




In [3]:
injection_file = '../injections/general_pop/injection_10.hdf'
test_injection = h5py.File(injection_file,'r')

for p in test_injection.keys():
    print(p, np.array(test_injection[p]))
for p in test_injection.attrs['static_args']:
    print(p,test_injection.attrs[p])

coa_phase [1.24446568]
dec [0.54809407]
distance [859.07896107]
inclination [2.09163498]
mass1 [39.70942895]
mass2 [5.93383772]
polarization [3.13221194]
ra [1.41243898]
approximant IMRPhenomXPHM
f_ref 20.0
f_lower 20.0
tc 1420871141.223


In [4]:
cp = create_parser(injection_file=injection_file,
                        variable_params_file='../config_files/var_par.ini',
                        data_file='../config_files/data_config/data_noise.ini',
                        model_file='../config_files/model_config/hm_model.ini',
                        mode_array='22 33',weighted_mode_peak = True)



In [5]:
hm = read_from_config(cp)


In [6]:
hm.update(coa_phase = 0)


In [7]:
print(hm.loglr)


using weighted peak
using analytic approximation
using weighted peak
0.00021641701459884644
87.05050182567794


In [8]:
print(hm.peaks)

[1.27061573 4.41220839]


In [12]:
shm = hm.shm

In [13]:
sh2 = np.abs(shm[2])
sh3 = np.abs(shm[3])
print(sh2, sh3)


168.49617923507182 13.989104350245348


In [15]:
n2 = np.arange(4)
phi2 = 0.5*(np.arctan(-shm[2].imag/shm[2].real) + n2*np.pi)
phi2[phi2 < 0] += 2 * np.pi
n3 = np.arange(6)
phi3 = (1/3)*(np.arctan(-shm[3].imag/shm[3].real) + n3*np.pi)
phi3[phi3 < 0] += 2*np.pi
print('phi2', phi2)
print('phi3',phi3)

phi2 [5.98531425 1.27292526 2.84372159 4.41451792]
phi3 [0.1956003  1.24279785 2.2899954  3.33719295 4.3843905  5.43158805]


In [16]:
diff = np.abs(phi2[:,None] - phi3[None,:])
i2,i3 = np.where(diff < 0.2)
weighted_peak = (np.abs(shm[2])*phi2[i2] + np.abs(shm[3])*phi3[i3])/(np.abs(shm[2])+np.abs(shm[3]))
print(weighted_peak)

[1.27061573 4.41220839]
